In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import string
import re

# Парсим текст "Преступления и наказания" и "Анны Карениной"

In [ ]:
def get_pages(url):
  page = requests.get(url+'index.html')
  if page.status_code == 200:
    print(f"...Страница {url} доступна...")
    soup = BeautifulSoup(page.text, "html.parser")
    pages = []
    for link in soup.find_all(class_ = 'go'):
      pages.append("https://ilibrary.ru" + link.find('a').get('href'))
    return pages

def parse_pages(pages):
    text = []
    for link in pages:
      page = requests.get(link)
      page_blocks = []
      if page.status_code == 200:
        soup = BeautifulSoup(page.text, "html.parser")
        for block in soup.find_all('z'):
          page_blocks.append(block.get_text())
        page_text = ''.join(page_blocks)
      text.append(page_text)
    full_text = ''.join(text)
    return full_text

def split_text_to_words(text):
  sentences = [sentence.strip() for sentence in text.split(".") if sentence.strip() != '']
  sentences = [re.findall(r'[ЁёА-я]+', sentence) for sentence in sentences]
  sentences = [[word.lower().strip() for word in sentence] for sentence in sentences]
  return sentences

def prepare_dataframe(author, url):
  print(f"...Сбор текста для {author}...")
  pages = get_pages(url)
  full_text = parse_pages(pages)
  sentences = split_text_to_words(full_text)
  df = pd.DataFrame(zip(sentences, [author]*len(sentences)), columns = ['sentence', 'author'])
  return df

def collect_dataset(dfs):
  dataset = pd.concat(dfs, axis = 0).reset_index(drop = True)
  return dataset

In [ ]:
dataframes = []

url_dict = {
    "Достоевский" : "https://ilibrary.ru/text/69/",
    "Толстой" : "https://ilibrary.ru/text/1099/"
}

for author, url in url_dict.items():
  df = prepare_dataframe(author, url)
  dataframes.append(df)

data = collect_dataset(dataframes)

...Сбор текста для Достоевский...
...Страница https://ilibrary.ru/text/69/ доступна...
...Сбор текста для Толстой...
...Страница https://ilibrary.ru/text/1099/ доступна...


In [ ]:
data.head()

,sentence,author
0,"[в, начале, июля, в, чрезвычайно, жаркое, врем...",Достоевский
1,"[он, благополучно, избегнул, встречи, с, своею...",Достоевский
2,"[каморка, его, приходилась, под, самою, кровле...",Достоевский
3,"[квартирная, же, хозяйка, его, у, которой, он,...",Достоевский
4,"[и, каждый, раз, молодой, человек, проходя, ми...",Достоевский


In [ ]:
# Сохраняем в pkl
data.to_pickle("text_data.pkl")